In [1]:
import sys
sys.path.append('/home/trukhinmaksim/src')
import json
from time import time
from collections import defaultdict
from numpy import array
from random import random
import numpy as np
from gensim.models.doc2vec import TaggedDocument

In [2]:
from src.utils.Corpus import Factory_21_04_25_HIGH as CorpusFactory
from src.utils.CacheAdapter import FlatAdapter, EXP_END_OF_DATA
from src.utils.Evaluator import Evaluator

testCorpus = CorpusFactory.createFlatTestCorpus()
trainCorpus = CorpusFactory.createFlatTrainCorpus()

In [5]:


relatedAdapter = FlatAdapter("/home/trukhinmaksim/src/data/cache_21-04-25/related_pairs_idx_21-04-25")
unrelatedAdapter = FlatAdapter("/home/trukhinmaksim/src/data/cache_21-04-25/unrelated_pairs_idx_21-04-25")

def isSimilar(doc1, doc2):
    if len(set(doc1.tags[2:]) & set(doc2.tags[2:])) >= 2:
        return True
    return False

def saveToAdapter(pairs, adapter):
    adapter.save(pairs)

In [6]:
N = 410172#72383
M = 1_538_100

def count(corpus):
    relatedPairs = []
    unrelatedPairs = []
    relatedCounter = 0
    unrelatedCounter = 0

    for i in range(N):
        if len(corpus[[i]][0].tags) < 2: continue
        for j in range(i + 1, N):
            if len(corpus[[j]][0].tags) < 2: continue
            if (i * N + j) % 5_000_000 == 0: print(f"Scanned {i * N + j} pairs")

            doc1, doc2 = corpus[[i, j]]
            if isSimilar(doc1, doc2):
                if relatedCounter < M:
                    #relatedPairs.append({"doc1" : i, "doc2" : j, "label" : 1})
                    relatedCounter += 1

            else:
                if unrelatedCounter < M:
                    #unrelatedPairs.append({"doc1" : i, "doc2" : j, "label" : 0})
                    unrelatedCounter += 1
                #c += 1

            if relatedCounter >= M and unrelatedCounter >= M:
                print(f"Found enough pairs, i = {i}, j = {j}")
                return

count(trainCorpus)

Scanned 10000000 pairs


KeyboardInterrupt: 

In [ ]:
memorized = set()

def countRandom():
    relatedPairs = []
    unrelatedPairs = []
    relatedCounter = 0
    unrelatedCounter = 0

    for i in range(N):
        if len(trainCorpus[[i]][0].tags) < 2: continue
        for j in range(i + 1, N):
            if len(trainCorpus[[j]][0].tags) < 2: continue
            if (i * N + j) % 1000000 == 0: print(f"Scanned {i * N + j} pairs")

            doc1, doc2 = trainCorpus[[i, j]]
            if isSimilar(doc1, doc2):
                if relatedCounter < 100_000:
                    relatedPairs.append({"doc1" : i, "doc2" : j, "label" : 1})
                    relatedCounter += 1

            else:
                if unrelatedCounter < 100_000:
                    unrelatedPairs.append({"doc1" : i, "doc2" : j, "label" : 0})
                    unrelatedCounter += 1
                #c += 1

            if relatedCounter >= 100_000 and unrelatedCounter >= 100_000:
                print(f"Found enough pairs, i = {i}, j = {j}")
                return

countRandom()

In [ ]:
def countAndSave():
    relatedPairs = []
    unrelatedPairs = []
    relatedCounter = 0
    unrelatedCounter = 0

    for i in range(N):
        if len(testCorpus[[i]][0].tags) < 2: continue
        for j in range(i + 1, N):
            if len(testCorpus[[j]][0].tags) < 2: continue
            if (i * N + j) % 1000000 == 0: print(f"Scanned {i * N + j} pairs")

            doc1, doc2 = testCorpus[[i, j]]
            if isSimilar(doc1, doc2):
                if relatedCounter < 100_000 and random() > 0.95:
                    relatedPairs.append({"doc1" : i, "doc2" : j, "label" : 1})
                    relatedCounter += 1

                    if len(relatedPairs) >= 10000:
                        saveToAdapter(relatedPairs, relatedAdapter)
                        relatedPairs.clear()
            else:
                if unrelatedCounter < 100_000 and random() > 0.99:
                    unrelatedPairs.append({"doc1" : i, "doc2" : j, "label" : 0})
                    unrelatedCounter += 1

                    if len(unrelatedPairs) >= 10000:
                        saveToAdapter(unrelatedPairs, unrelatedAdapter)
                        unrelatedPairs.clear()
                #c += 1

            if relatedCounter >= 100_000 and unrelatedCounter >= 100_000:
                print(f"Found enough pairs, i = {i}, j = {j}")
                saveToAdapter(unrelatedPairs, unrelatedAdapter)
                saveToAdapter(relatedPairs, relatedAdapter)
                return

countAndSave()

In [4]:
relatedAdapter.reset()
unrelatedAdapter.reset()

relC = 0
unrelC = 0

def checkDataAmount(adapter):
    counter = 0
    while True:
        try:
            counter += 1
            adapter.load(1)
        except EXP_END_OF_DATA:
            return counter

print(checkDataAmount(relatedAdapter))
print(checkDataAmount(unrelatedAdapter))

100001
100001


In [15]:
relatedAdapter = FlatAdapter("/home/trukhinmaksim/src/data/cache_21-04-25/related_pairs_idx_21-04-25")
unrelatedAdapter = FlatAdapter("/home/trukhinmaksim/src/data/cache_21-04-25/unrelated_pairs_idx_21-04-25")

class Mod:
    def infer_vector(self, doc):
        print(doc.words)
        return np.array([0, 0])

class RelAda:
    def load(self, amount):
        return [TaggedDocument(words = "q w e".split(), tags = [1]), TaggedDocument(words = "q e".split(), tags = [2]), TaggedDocument(words = "w e o".split(), tags = [3])]

class UnrelAda:
    def load(self, amount):
        return [TaggedDocument(words = "q w e".split(), tags = [1]), TaggedDocument(words = "q e".split(), tags = [2]), TaggedDocument(words = "w e o".split(), tags = [3])]



v = [0.99, 0.001, 0.002, 0.98, 0.97, 0.01]
i = -1

def sim(v1, v2):
    global i
    if i == 4: return 0
    i += 1
    return v[i]

evaluator = Evaluator(relatedAdapter, unrelatedAdapter, testCorpus)
evaluator.setModel(Mod())
evaluator.setSimilarityCheck(sim)
evaluator.statisticalTest(v[:3], v[3:])


0.8